# North Star Supply Chain — Synthetic Dataset Generation

This notebook creates a reproducible synthetic supply-chain dataset for portfolio analysis. It models product demand, supplier reliability, warehouse allocation, replenishment purchasing, and daily inventory movement.

In [ ]:
from pathlib import Path
from datetime import date, timedelta
import random

import numpy as np
import pandas as pd

# Reproducibility and safe export controls
SEED = 20250929
WRITE_CSV = False

# Start Jupyter from the repository root so these relative paths resolve correctly.
PROJECT_ROOT = Path.cwd().resolve()
RAW_DATA_PATH = PROJECT_ROOT / 'data' / 'raw'

random.seed(SEED)
rng = np.random.default_rng(SEED)

print(f'Python/pandas/numpy environment: pandas {pd.__version__}, numpy {np.__version__}')
print(f'Seed: {SEED}; CSV export enabled: {WRITE_CSV}')

## 1. Scenario parameters

The model uses 2025 as the sales period and keeps a January 2026 calendar buffer for late purchase deliveries.

In [ ]:
TRANSACTION_START = date(2025, 1, 1)
TRANSACTION_END = date(2025, 12, 31)
MAX_LEAD_TIME_DAYS = 21
MAX_DELIVERY_DELAY_DAYS = 10
CALENDAR_END = TRANSACTION_END + timedelta(days=MAX_LEAD_TIME_DAYS + MAX_DELIVERY_DELAY_DAYS)

PRODUCT_COUNT = 120
SUPPLIER_COUNT = 20
WAREHOUSE_NAMES = [
    'San Francisco DC', 'Los Angeles DC', 'Dallas DC', 'Chicago DC', 'New York DC'
]
CATEGORIES = [
    'Electronics', 'Home Appliances', 'Kitchen', 'Furniture', 'Personal Care',
    'Sports & Fitness', 'Office Supplies', 'Outdoor', 'Automotive', 'Pet Supplies'
]
DEMAND_RANGES = {'High': (25, 60), 'Medium': (10, 25), 'Low': (1, 8)}
DEMAND_PROFILES = ['High', 'Medium', 'Low']
DEMAND_WEIGHTS = [0.20, 0.50, 0.30]
SEASONALITY = {1: 0.95, 2: 0.95, 3: 1.00, 4: 1.00, 5: 1.00, 6: 1.05,
               7: 1.05, 8: 1.00, 9: 1.00, 10: 1.05, 11: 1.15, 12: 1.20}
RELIABILITY_PROFILES = ['Reliable', 'Moderate', 'Unreliable']
RELIABILITY_WEIGHTS = [0.60, 0.25, 0.15]
DELIVERY_PROBABILITIES = {
    'Reliable': [0.90, 0.08, 0.02],
    'Moderate': [0.70, 0.20, 0.10],
    'Unreliable': [0.40, 0.35, 0.25],
}

## 2. Master data

`demand_profile` and `reliability_profile` are generator controls. They remain in memory for modelling but are deliberately excluded from the raw database exports.

In [ ]:
categories_df = pd.DataFrame({
    'category_id': [f'CAT{i:03d}' for i in range(1, len(CATEGORIES) + 1)],
    'category_name': CATEGORIES,
})

products = []
for i in range(1, PRODUCT_COUNT + 1):
    unit_cost = round(random.uniform(5, 250), 2)
    selling_price = round(unit_cost * random.uniform(1.2, 2.0), 2)
    products.append({
        'product_id': f'P{i:03d}',
        'product_name': f'Product {i:03d}',
        'category_id': random.choice(categories_df['category_id'].tolist()),
        'demand_profile': random.choices(DEMAND_PROFILES, weights=DEMAND_WEIGHTS, k=1)[0],
        'unit_cost': unit_cost,
        'selling_price': selling_price,
        'holding_cost_per_unit': round(unit_cost * random.uniform(0.01, 0.05), 2),
        'moq': random.choice([10, 25, 50, 100, 200]),
    })
products_df = pd.DataFrame(products)
products_df['gross_margin'] = products_df['selling_price'] - products_df['unit_cost']

supplier_locations = ['California', 'Texas', 'New York', 'Illinois', 'Florida', 'New Jersey', 'Georgia', 'Ohio']
suppliers = []
for i in range(1, SUPPLIER_COUNT + 1):
    suppliers.append({
        'supplier_id': f'S{i:03d}',
        'supplier_name': f'Supplier {i:03d}',
        'supplier_location': random.choice(supplier_locations),
        'standard_lead_time_days': random.randint(3, MAX_LEAD_TIME_DAYS),
        'reliability_profile': random.choices(RELIABILITY_PROFILES, weights=RELIABILITY_WEIGHTS, k=1)[0],
    })
suppliers_df = pd.DataFrame(suppliers)

warehouses_df = pd.DataFrame({
    'warehouse_id': [f'W{i:03d}' for i in range(1, len(WAREHOUSE_NAMES) + 1)],
    'warehouse_name': WAREHOUSE_NAMES,
})

In [ ]:
calendar_dates = pd.date_range(TRANSACTION_START, CALENDAR_END, freq='D')
transaction_dates = pd.date_range(TRANSACTION_START, TRANSACTION_END, freq='D')
date_df = pd.DataFrame({'date': calendar_dates})
date_df['month'] = date_df['date'].dt.month_name()
date_df['quarter'] = 'Q' + date_df['date'].dt.quarter.astype(str)
date_df['year'] = date_df['date'].dt.year

assert (products_df['selling_price'] > products_df['unit_cost']).all()
assert date_df['date'].min() == pd.Timestamp(TRANSACTION_START)
assert date_df['date'].max() == pd.Timestamp(CALENDAR_END)

## 3. Product–supplier relationships

Every product has one primary supplier and zero to two eligible alternatives. Supplier-specific costs vary modestly around the product’s standard cost.

In [ ]:
supplier_ids = suppliers_df['supplier_id'].tolist()
product_suppliers = []
for product_id in products_df['product_id']:
    primary_supplier = random.choice(supplier_ids)
    product_suppliers.append({'product_id': product_id, 'supplier_id': primary_supplier, 'is_primary_supplier': True})

    additional_count = random.choices([0, 1, 2], weights=[0.30, 0.50, 0.20], k=1)[0]
    alternatives = random.sample([supplier for supplier in supplier_ids if supplier != primary_supplier], additional_count)
    product_suppliers.extend(
        {'product_id': product_id, 'supplier_id': supplier, 'is_primary_supplier': False}
        for supplier in alternatives
    )

product_suppliers_df = pd.DataFrame(product_suppliers)
cost_lookup = products_df.set_index('product_id')['unit_cost'].to_dict()
product_suppliers_df['supplier_unit_cost'] = (
    product_suppliers_df['product_id'].map(cost_lookup)
    * rng.uniform(0.90, 1.10, len(product_suppliers_df))
).round(2)

assert not product_suppliers_df.duplicated(['product_id', 'supplier_id']).any()
assert (product_suppliers_df.groupby('product_id')['is_primary_supplier'].sum() == 1).all()

## 4. Demand and warehouse allocation

Daily product demand is seasonal and noisy. Integer warehouse allocation uses a largest-remainder method so each product-day’s allocated demand exactly equals its total planned demand.

In [ ]:
product_demand_df = products_df[['product_id', 'demand_profile']].copy()
product_demand_df['baseline_daily_demand'] = product_demand_df['demand_profile'].map(
    lambda profile: round(random.uniform(*DEMAND_RANGES[profile]), 2)
)
demand_baseline = product_demand_df.set_index('product_id')['baseline_daily_demand'].to_dict()

BASE_ALLOCATION = 0.05
remaining_allocation = 1 - BASE_ALLOCATION * len(warehouses_df)
allocation_rows = []
for product_id in products_df['product_id']:
    weights = BASE_ALLOCATION + remaining_allocation * rng.dirichlet(np.ones(len(warehouses_df)))
    allocation_rows.extend(
        {'product_id': product_id, 'warehouse_id': warehouse_id, 'allocation_pct': weight}
        for warehouse_id, weight in zip(warehouses_df['warehouse_id'], weights)
    )
warehouse_allocation_df = pd.DataFrame(allocation_rows)

planned_demand_rows = []
for product_id in products_df['product_id']:
    baseline = demand_baseline[product_id]
    for sale_date in transaction_dates:
        quantity = max(0, round(random.gauss(baseline * SEASONALITY[sale_date.month], baseline * SEASONALITY[sale_date.month] * 0.20)))
        planned_demand_rows.append({'product_id': product_id, 'sale_date': sale_date, 'planned_quantity': quantity})
planned_product_demand_df = pd.DataFrame(planned_demand_rows)

def allocate_integer(total, weights):
    raw = np.asarray(weights) * total
    allocated = np.floor(raw).astype(int)
    remainder = int(total - allocated.sum())
    if remainder:
        ranked = np.argsort(-(raw - allocated), kind='stable')
        allocated[ranked[:remainder]] += 1
    return allocated

allocation_lookup = warehouse_allocation_df.set_index(['product_id', 'warehouse_id'])['allocation_pct'].to_dict()
warehouse_ids = warehouses_df['warehouse_id'].tolist()
daily_demand_rows = []
for row in planned_product_demand_df.itertuples(index=False):
    weights = [allocation_lookup[(row.product_id, warehouse_id)] for warehouse_id in warehouse_ids]
    quantities = allocate_integer(row.planned_quantity, weights)
    daily_demand_rows.extend(
        {'sale_date': row.sale_date, 'product_id': row.product_id, 'warehouse_id': warehouse_id, 'warehouse_demand': quantity}
        for warehouse_id, quantity in zip(warehouse_ids, quantities)
    )
daily_demand_df = pd.DataFrame(daily_demand_rows)

assert np.isclose(warehouse_allocation_df.groupby('product_id')['allocation_pct'].sum(), 1).all()
assert daily_demand_df['warehouse_demand'].sum() == planned_product_demand_df['planned_quantity'].sum()

## 5. Opening inventory and replenishment purchases

Purchases are demand-driven, respect MOQ, use valid suppliers, and have delivery delays based on supplier reliability. The January calendar buffer keeps late deliveries inside the date dimension.

In [ ]:
warehouse_demand_df = warehouse_allocation_df.merge(
    product_demand_df[['product_id', 'baseline_daily_demand']], on='product_id'
)
warehouse_demand_df['expected_daily_demand'] = (
    warehouse_demand_df['baseline_daily_demand'] * warehouse_demand_df['allocation_pct']
)
starting_inventory = warehouse_demand_df[['product_id', 'warehouse_id', 'expected_daily_demand']].copy()
starting_inventory['coverage_days'] = rng.integers(20, 46, len(starting_inventory))
starting_inventory['opening_stock'] = (
    starting_inventory['expected_daily_demand'] * starting_inventory['coverage_days']
).round().astype(int)

product_moq = products_df.set_index('product_id')['moq'].to_dict()
supplier_cost = product_suppliers_df.set_index(['product_id', 'supplier_id'])['supplier_unit_cost'].to_dict()
supplier_info = suppliers_df.set_index('supplier_id').to_dict('index')
eligible_suppliers = product_suppliers_df.groupby('product_id')['supplier_id'].apply(list).to_dict()

purchase_rows = []
purchase_counter = 1
latest_purchase_date = pd.Timestamp(CALENDAR_END) - pd.Timedelta(days=MAX_LEAD_TIME_DAYS + MAX_DELIVERY_DELAY_DAYS)
for product_id in products_df['product_id']:
    purchase_date = pd.Timestamp(TRANSACTION_START) + pd.Timedelta(days=random.randint(15, 30))
    replenishment_interval = random.randint(18, 25)
    while purchase_date <= latest_purchase_date:
        coverage_days = random.randint(25, 35)
        target_quantity = demand_baseline[product_id] * coverage_days * random.uniform(0.85, 1.15)
        moq = product_moq[product_id]
        quantity = int(np.ceil(target_quantity / moq) * moq)
        supplier_id = random.choice(eligible_suppliers[product_id])
        supplier = supplier_info[supplier_id]
        delivery_class = random.choices(['on_time', 'slightly_late', 'very_late'], weights=DELIVERY_PROBABILITIES[supplier['reliability_profile']], k=1)[0]
        delay_days = 0 if delivery_class == 'on_time' else random.randint(1, 3) if delivery_class == 'slightly_late' else random.randint(4, 10)
        expected_delivery_date = purchase_date + pd.Timedelta(days=supplier['standard_lead_time_days'])
        actual_delivery_date = expected_delivery_date + pd.Timedelta(days=delay_days)
        purchase_weights = [allocation_lookup[(product_id, warehouse_id)] for warehouse_id in warehouse_ids]
        purchase_rows.append({
            'purchase_id': f'PO{purchase_counter:05d}', 'product_id': product_id, 'supplier_id': supplier_id,
            'purchase_date': purchase_date, 'quantity': quantity, 'unit_cost': supplier_cost[(product_id, supplier_id)],
            'expected_delivery_date': expected_delivery_date, 'actual_delivery_date': actual_delivery_date,
            'warehouse_id': random.choices(warehouse_ids, weights=purchase_weights, k=1)[0],
        })
        purchase_counter += 1
        purchase_date += pd.Timedelta(days=replenishment_interval)

purchases_df = pd.DataFrame(purchase_rows)
assert (purchases_df['quantity'] >= purchases_df['product_id'].map(product_moq)).all()
assert purchases_df['actual_delivery_date'].max() <= pd.Timestamp(CALENDAR_END)

## 6. Daily inventory simulation and fulfilled sales

Planned demand becomes fulfilled sales only when stock is available. Unfulfilled demand is retained as `stockout_quantity` in inventory.

In [ ]:
demand_lookup = daily_demand_df.set_index(['sale_date', 'product_id', 'warehouse_id'])['warehouse_demand'].to_dict()
receipt_lookup = purchases_df.groupby(['actual_delivery_date', 'product_id', 'warehouse_id'])['quantity'].sum().to_dict()
starting_stock_lookup = starting_inventory.set_index(['product_id', 'warehouse_id'])['opening_stock'].to_dict()

previous_closing_stock = {}
inventory_rows = []
for current_date in calendar_dates:
    for product_id in products_df['product_id']:
        for warehouse_id in warehouse_ids:
            opening_stock = (
                starting_stock_lookup[(product_id, warehouse_id)]
                if current_date == calendar_dates[0]
                else previous_closing_stock[(product_id, warehouse_id)]
            )
            received_quantity = receipt_lookup.get((current_date, product_id, warehouse_id), 0)
            demand_quantity = demand_lookup.get((current_date, product_id, warehouse_id), 0)
            sold_quantity = min(demand_quantity, opening_stock + received_quantity)
            closing_stock = opening_stock + received_quantity - sold_quantity
            inventory_rows.append({
                'date': current_date, 'product_id': product_id, 'warehouse_id': warehouse_id,
                'opening_stock': opening_stock, 'received_quantity': received_quantity,
                'demand_quantity': demand_quantity, 'sold_quantity': sold_quantity,
                'stockout_quantity': demand_quantity - sold_quantity, 'closing_stock': closing_stock,
            })
            previous_closing_stock[(product_id, warehouse_id)] = closing_stock

inventory_df = pd.DataFrame(inventory_rows)
sales_df = (
    inventory_df.loc[inventory_df['sold_quantity'] > 0]
    .groupby(['date', 'product_id'], as_index=False)['sold_quantity'].sum()
    .rename(columns={'date': 'sale_date', 'sold_quantity': 'quantity'})
    .sort_values(['sale_date', 'product_id'])
    .reset_index(drop=True)
)
sales_df.insert(0, 'sale_id', [f'SAL{i:06d}' for i in range(1, len(sales_df) + 1)])

## 7. Validation

These checks validate the internally consistent records before any future, documented data-quality issues are injected.

In [ ]:
inventory_balance_error = (
    inventory_df['opening_stock'] + inventory_df['received_quantity']
    - inventory_df['sold_quantity'] - inventory_df['closing_stock']
).abs().max()
stockout_balance_error = (
    inventory_df['demand_quantity'] - inventory_df['sold_quantity'] - inventory_df['stockout_quantity']
).abs().max()

checks = {
    'unique product IDs': not products_df['product_id'].duplicated().any(),
    'unique supplier IDs': not suppliers_df['supplier_id'].duplicated().any(),
    'unique sale IDs': not sales_df['sale_id'].duplicated().any(),
    'unique purchase IDs': not purchases_df['purchase_id'].duplicated().any(),
    'valid purchase product IDs': purchases_df['product_id'].isin(products_df['product_id']).all(),
    'valid purchase supplier IDs': purchases_df['supplier_id'].isin(suppliers_df['supplier_id']).all(),
    'valid purchase warehouse IDs': purchases_df['warehouse_id'].isin(warehouses_df['warehouse_id']).all(),
    'inventory key uniqueness': not inventory_df.duplicated(['date', 'product_id', 'warehouse_id']).any(),
    'non-negative inventory': (inventory_df[['opening_stock', 'received_quantity', 'sold_quantity', 'stockout_quantity', 'closing_stock']] >= 0).all().all(),
    'inventory balance': inventory_balance_error == 0,
    'stockout balance': stockout_balance_error == 0,
    'demand reconciliation': daily_demand_df['warehouse_demand'].sum() == inventory_df['demand_quantity'].sum(),
    'fulfilled-sales reconciliation': sales_df['quantity'].sum() == inventory_df['sold_quantity'].sum(),
}

assert all(checks.values()), checks
print(pd.Series({
    'categories': len(categories_df), 'products': len(products_df), 'suppliers': len(suppliers_df),
    'warehouses': len(warehouses_df), 'calendar dates': len(date_df),
    'product-supplier pairs': len(product_suppliers_df), 'sales': len(sales_df),
    'purchases': len(purchases_df), 'inventory': len(inventory_df),
}).to_string())
print(f'All {len(checks)} validation checks passed.')
print(f'Planned demand: {daily_demand_df.warehouse_demand.sum():,}; fulfilled sales: {sales_df.quantity.sum():,}; stockouts: {inventory_df.stockout_quantity.sum():,}')

## 8. Export schema

The exports deliberately omit internal generator controls. Do not enable export unless you intend to replace `data/raw/*.csv` and reload downstream database tables.

In [ ]:
exports = {
    'categories': categories_df,
    'products': products_df[['product_id', 'product_name', 'category_id', 'unit_cost', 'selling_price', 'holding_cost_per_unit', 'moq']],
    'suppliers': suppliers_df[['supplier_id', 'supplier_name', 'supplier_location', 'standard_lead_time_days']],
    'warehouses': warehouses_df,
    'date': date_df,
    'product_suppliers': product_suppliers_df,
    'sales': sales_df,
    'purchases': purchases_df[['purchase_id', 'product_id', 'supplier_id', 'purchase_date', 'quantity', 'unit_cost', 'expected_delivery_date', 'actual_delivery_date', 'warehouse_id']],
    'inventory': inventory_df,
}

if WRITE_CSV:
    RAW_DATA_PATH.mkdir(parents=True, exist_ok=True)
    for table, dataframe in exports.items():
        dataframe.to_csv(RAW_DATA_PATH / f'{table}.csv', index=False)
    print(f'Wrote {len(exports)} CSV files to {RAW_DATA_PATH}')
else:
    print('CSV export skipped. Set WRITE_CSV = True only when intentionally regenerating data/raw.')